In [1]:
# Sistema completo a due stadi, riemanniano a banco di filtri con riallineamento per run,
# su tutte e sei le run di motor imagery. Copia di RiemannFB_rest_active_imagery con le sole
# modifiche necessarie ad aggiungere il secondo stadio.
#
#   Stadio 1 -> riposo contro attivazione, addestrato su tutte le finestre.
#   Stadio 2 -> quale movimento, addestrato sulle sole finestre attive: pugno sinistro,
#               pugno destro, entrambi i pugni, entrambi i piedi.
#
# Le prime due classi vivono nelle run 4, 8, 12 (TASK2), le altre due nelle run 6, 10, 14
# (TASK4). Nessuna run contiene tutte e quattro, quindi ogni fold tiene fuori UNA RUN PER
# FAMIGLIA e testa sulla loro unione: {12, 14}, poi {8, 10}, poi {4, 6}, addestrando ogni volta
# sulle quattro rimanenti. Solo cosi' l'insieme di test contiene tutte e cinque le classi finali
# e la run di test resta comunque una sessione mai vista.
#
# Le due probabilita' si compongono in una distribuzione su cinque esiti:
#     P(riposo) = 1 - P(attivo)          P(movimento k) = P(attivo) x P(k | attivo)
# che somma a uno, e su cui si applica la stessa soglia probabilistica delle altre pipeline.
# E' una cascata "morbida": lo stadio 1 non decide da solo ma pesa le alternative dello stadio 2,
# cosi' un primo stadio incerto non produce un errore irrecuperabile ma abbassa la confidenza
# complessiva, e la finestra viene scartata. E' il comportamento voluto in una BCI.
#
# Oltre al risultato della catena vengono riportate le accuratezze dei due stadi presi
# separatamente, lo stadio 2 valutato sulle finestre davvero attive cioe' come se lo stadio 1
# non sbagliasse mai: e' il confronto che misura quanto costa metterli in serie.
# Le run di motor execution restano fuori di proposito: il movimento reale porta con se'
# attivita' muscolare, e un rilevatore addestrato anche su quelle imparerebbe in buona parte a
# riconoscere i muscoli invece della corteccia.
#
# Perche' l'approccio riemanniano e non il CSP: riposo contro attivazione non e' un contrasto
# spaziale ma una differenza di livello, perche' il ritmo mu cala su entrambi gli emisferi.
# Le feature del CSP sono log-potenze assolute e per riconoscere "meno potenza del solito"
# manca loro il riferimento a cosa sia il solito, che varia molto fra soggetti e fra sessioni.
# Il riallineamento per run costruisce esattamente quel riferimento: dopo la ricentratura ogni
# finestra e' descritta rispetto al comportamento medio della sua stessa run. E dato che due
# terzi delle finestre di una run sono riposo, il baricentro coincide di fatto con lo stato di
# riposo, quindi le finestre attive sono quelle che se ne allontanano.
# --- Preambolo standard ------------------------------------------------------
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "dataset_description.json").exists()
)
DATA  = PROJECT_ROOT / "data"
PLOTS = PROJECT_ROOT / "src" / "plots"
sys.path.insert(0, str(PROJECT_ROOT / "src"))
# -----------------------------------------------------------------------------

import time
import warnings
import numpy as np
import mne
from mne_bids import BIDSPath, read_raw_bids
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.exceptions import ConvergenceWarning
from util.preprocessing import create_sliding_windows, create_window_labels
from util.riemann_filterbank import (
    DEFAULT_BANDS, run_covariances, reference_mean, recenter, to_tangent, n_features
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)
mne.set_log_level('WARNING')

# --- Configurazione ----------------------------------------------------------

root = DATA
runs = ["4", "8", "12", "6", "10", "14"]

# Ogni fold tiene fuori una run per famiglia e testa sull'unione delle due.
test_pairs = [("12", "14"), ("8", "10"), ("4", "6")]

# Il prefisso dell'annotazione dipende dal compito della run: le run sinistra/destra usano
# TASK2, quelle pugni/piedi TASK4. In entrambe T0 e' il riposo e T1, T2 le due classi attive,
# che pero' significano movimenti diversi nelle due famiglie e vanno quindi rinumerate.
TASK_PREFIX = {"4": "TASK2", "8": "TASK2", "12": "TASK2",
               "6": "TASK4", "10": "TASK4", "14": "TASK4"}

# Etichette finali del sistema: lo 0 e' il riposo, da 1 a 4 i movimenti.
CLASS_NAMES = ["riposo", "pugno sx", "pugno dx", "due pugni", "due piedi"]
first_person = 1
people = 109

window_size = 2         # identici alle altre pipeline, per confrontabilita'
step_size = 0.5
label_threshold = 0.8

# Banda larga: le sotto-bande del banco fanno la selezione fine
L_FREQ = 4
H_FREQ = 40
BANDS = DEFAULT_BANDS   # [(8, 13), (13, 20), (20, 30)]

CHANNELS = ["C3", "C4", "Cz", "Fc3", "Fc4", "Fcz", "Cp3", "Cp4", "Cpz"]

# Le finestre a cavallo fra un trial attivo e il riposo successivo non raggiungono l'80% di
# nessuna classe e vengono etichettate riposo per esclusione, pur contenendo movimento
# immaginato. Nella pipeline sinistra/destra finivano fra gli scarti insieme al riposo; qui il
# riposo e' una classe, quindi se restassero ne inquinerebbero le etichette.
DROP_BOUNDARY = True

# Riallineamento delle covarianze:
#   "per_run" -> ogni run viene riportata sul proprio baricentro. Il baricentro non usa le
#                etichette, quindi si puo' stimare anche sulla run di test: e' adattamento di
#                dominio non supervisionato. E' la configurazione che ci si aspetta migliore.
#   "train"   -> tutte le run vengono riportate sul baricentro delle sole run di training.
#                Piu' conservativo: nessuna statistica della run di test viene usata.
RECENTER = "per_run"

# Il classificatore e' quasi privo di iperparametri: la LDA con shrinkage di Ledoit-Wolf
# regolarizza da sola ed e' deterministica, a differenza dell'SVM con probability=True le cui
# probabilita' passano da una calibrazione di Platt con mescolamento casuale interno.
# Griglia minuscola: con 28 trial indipendenti cercare fra decine di combinazioni insegue rumore.
param_grid = {
    "clf__shrinkage": ["auto", 0.2, 0.5],
}

# Come si passa dalle due probabilita' alla risposta finale:
#   "cascata"      -> si decide in sequenza. Prima se c'e' un movimento, confrontando
#                     P(attivo) con 0.5; poi, solo in caso affermativo, quale dei quattro.
#                     Ogni decisione usa il riferimento coerente con la balanced accuracy
#                     con cui poi la misuriamo.
#   "composizione" -> si prende il massimo delle cinque probabilita' composte. E' corretto
#                     come probabilita', ma sbilanciato verso il riposo: il riposo e' un
#                     esito unico e non divide la sua probabilita' con nessuno, mentre i
#                     quattro movimenti se la spartiscono. Perche' un movimento vinca serve
#                     P(attivo) > 1 / (1 + il migliore dei quattro), cioe' circa il 70%.
DECISION = "cascata"

START_THRESHOLD = 0.90

# Il livello del caso: sotto non ha senso scendere. Con due classi vale 0.50, che e' il
# valore usato da tutte le altre pipeline; con cinque vale 0.20. Tenendo 0.50 anche qui si
# scarterebbero predizioni legittime, perche' la confidenza di un movimento e' un prodotto
# di due numeri minori di uno e sta tipicamente intorno a 0.3.
MIN_THRESHOLD = 1 / len(CLASS_NAMES)
MIN_ACCEPTED_RATIO = 0.70

# NaN e non 0: un fold che fallisce deve restare fuori dalle medie, non entrarci come 0%
all_accuracy = np.full((people, len(test_pairs)), np.nan)   # catena completa, cinque classi
all_discarded = np.full((people, len(test_pairs)), np.nan)
all_stage1 = np.full((people, len(test_pairs)), np.nan)     # solo riposo vs attivo
all_stage2 = np.full((people, len(test_pairs)), np.nan)     # solo i quattro movimenti
cm_sum = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)))

print(f"Feature per finestra: {n_features(len(CHANNELS), len(BANDS))} "
      f"({len(BANDS)} bande x {len(CHANNELS)} canali)")
print()

# --- Loop principale ---------------------------------------------------------

for i in range(first_person, first_person + people):
    subject = f"{i:03d}"
    subject_start = time.time()

    print("=" * 60)
    print(f"Paziente {subject}")
    print("=" * 60)

    # Lettura, finestre e covarianze dipendono solo dal segnale, non da quale run faccia da
    # test: si calcolano una volta sola per soggetto invece di ripeterle dentro ogni fold
    # della LORO. Per ogni run si conservano le covarianze di TUTTE le finestre (servono a
    # stimare il baricentro senza usare le etichette) e la maschera di quelle utilizzabili.
    per_run = {}
    trial_offset = 0

    for run in runs:
        bids_path = BIDSPath(
            subject=subject, task="motion", run=run, datatype="eeg", root=root,
        )

        try:
            raw = read_raw_bids(bids_path, verbose=False)
            events, event_id = mne.events_from_annotations(raw, verbose=False)
            raw.load_data(verbose=False)
            raw.filter(l_freq=L_FREQ, h_freq=H_FREQ, verbose=False)
            raw.set_eeg_reference('average', projection=False, verbose=False)

            sfreq = raw.info['sfreq']
            prefix = TASK_PREFIX[run]
            event_map = {
                event_id[f'{prefix}T0']: 1,
                event_id[f'{prefix}T1']: 2,
                event_id[f'{prefix}T2']: 3
            }

            windows, window_samples, step_samples, total_samples = create_sliding_windows(
                raw, window_size, step_size
            )
            picks = mne.pick_channels(raw.ch_names, CHANNELS)
            windows = windows[:, picks, :]

            y, groups = create_window_labels(
                events, event_map, total_samples, window_samples, step_samples,
                threshold=label_threshold, return_groups=True
            )
            groups = np.where(groups >= 0, groups + trial_offset, -1)
            trial_offset += len(events)

            # Covarianze su TUTTE le finestre: il baricentro va stimato sull'intera run,
            # senza guardare quali finestre siano attive ne' di che classe siano.
            covs = run_covariances(windows, sfreq, BANDS)

            # Anche il baricentro di ogni run e' indipendente dal fold, quindi si calcola qui.
            # Con RECENTER = "train" il riferimento dipende invece da quali run siano di
            # training e viene ricalcolato dentro il ciclo dei fold.
            refs = [reference_mean(c) for c in covs] if RECENTER == "per_run" else None

            # create_window_labels numera 1 il riposo e 2, 3 le due classi attive della run.
            # Le due famiglie usano gli stessi due codici per movimenti diversi, quindi si
            # rinumera su cinque etichette comuni a tutto il dataset: 0 riposo, 1 pugno sx,
            # 2 pugno dx, 3 due pugni, 4 due piedi.
            shift = 0 if prefix == "TASK2" else 2
            y5 = np.where(y == 1, 0, y - 1 + shift)

            # Le finestre col sentinella -1 cadono dopo l'ultimo evento della run, fuori da
            # ogni trial: verrebbero etichettate riposo per esclusione e non sono utilizzabili.
            keep = groups >= 0

            if DROP_BOUNDARY:
                # Una finestra di confine si riconosce dal fatto che e' etichettata riposo
                # ma il suo trial di maggioranza e' un trial attivo.
                active_trials = np.unique(groups[y != 1])
                keep &= ~((y == 1) & np.isin(groups, active_trials))

            per_run[run] = {
                "covs": covs,
                "ref": refs,
                "mask": keep,
                "y": y5[keep],
                "groups": groups[keep],
            }

        except Exception as e:
            print(f"Errore {subject} run {run}: {e}")

    if not all(r in per_run for r in runs):
        print(f"  Dati insufficienti: soggetto {subject} saltato\n")
        continue

    for test_index, test_runs in enumerate(test_pairs):
        test_start = time.time()
        train_runs = [run for run in runs if run not in test_runs]

        # --- Riallineamento ---------------------------------------------------
        # Se il riferimento e' quello di training, si calcola una volta sola dalle covarianze
        # delle run di training messe insieme; altrimenti ogni run usa il proprio, gia' pronto.
        if RECENTER == "train":
            references = [
                reference_mean(np.concatenate([per_run[r]["covs"][b] for r in train_runs]))
                for b in range(len(BANDS))
            ]

        features = {}
        for run in runs:
            aligned = []
            for b in range(len(BANDS)):
                covs_b = per_run[run]["covs"][b]
                ref = references[b] if RECENTER == "train" else per_run[run]["ref"][b]
                aligned.append(recenter(covs_b, ref))

            # Solo dopo il riallineamento si tengono le finestre utilizzabili
            mask = per_run[run]["mask"]
            features[run] = to_tangent([a[mask] for a in aligned])

        X_train = np.concatenate([features[r] for r in train_runs])
        y_train = np.concatenate([per_run[r]["y"] for r in train_runs])
        groups_train = np.concatenate([per_run[r]["groups"] for r in train_runs])

        # Le due run di test vengono unite: e' l'unico modo di avere tutte e cinque le classi
        # nello stesso insieme di valutazione.
        X_test = np.concatenate([features[r] for r in test_runs])
        y_test = np.concatenate([per_run[r]["y"] for r in test_runs])

        active_train = y_train > 0
        n_rest, n_active = int((~active_train).sum()), int(active_train.sum())

        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

        # --- Stadio 1: riposo contro attivazione -------------------------------
        # Le due classi non sono bilanciate: il riposo occupa circa il doppio del tempo.
        # Senza indicazioni la LDA userebbe le frequenze osservate come probabilita' a priori e
        # favorirebbe il riposo; imponendole uguali si ottiene l'equivalente dello
        # class_weight='balanced' usato nella versione CSP.
        pipe1 = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearDiscriminantAnalysis(solver="lsqr", priors=np.full(2, 0.5))),
        ])
        grid1 = GridSearchCV(pipe1, param_grid, cv=cv,
                             scoring="balanced_accuracy", n_jobs=-1)
        grid1.fit(X_train, (y_train > 0).astype(int), groups=groups_train)

        # --- Stadio 2: quale movimento -----------------------------------------
        # Addestrato sulle sole finestre attive, che sono quelle che in esercizio arriverebbero
        # fin qui. Il raggruppamento per trial va filtrato insieme ai dati, altrimenti gli
        # indici non corrisponderebbero piu'.
        pipe2 = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearDiscriminantAnalysis(solver="lsqr", priors=np.full(4, 0.25))),
        ])
        grid2 = GridSearchCV(pipe2, param_grid, cv=cv,
                             scoring="balanced_accuracy", n_jobs=-1)
        grid2.fit(X_train[active_train], y_train[active_train] - 1,
                  groups=groups_train[active_train])

        # --- La catena ----------------------------------------------------------
        # Le due probabilita' si compongono in una distribuzione sui cinque esiti finali.
        p_active = grid1.predict_proba(X_test)[:, 1]
        p_class = grid2.predict_proba(X_test)
        probs = np.column_stack([1 - p_active, p_active[:, None] * p_class])

        if DECISION == "cascata":
            # Prima se c'e' movimento, poi quale: lo stadio 2 sceglie fra quattro alternative
            # gia' selezionate, senza doversi misurare anche con il riposo.
            predictions = np.where(p_active >= 0.5, 1 + np.argmax(p_class, axis=1), 0)
        else:
            predictions = np.argmax(probs, axis=1)

        # La confidenza e' la probabilita' della classe effettivamente predetta. Con
        # "composizione" coincide con il massimo; con "cascata" puo' essere minore, perche'
        # la classe scelta non e' necessariamente quella con la probabilita' composta piu' alta.
        confidence = probs[np.arange(len(predictions)), predictions]

        threshold = START_THRESHOLD
        accepted_mask = confidence >= threshold
        while np.sum(accepted_mask) < MIN_ACCEPTED_RATIO * len(y_test) and threshold > MIN_THRESHOLD:
            threshold -= 0.05
            accepted_mask = confidence >= threshold

        accepted = int(np.sum(accepted_mask))
        total = len(y_test)
        discarded = total - accepted

        accuracy = balanced_accuracy_score(y_test[accepted_mask], predictions[accepted_mask])
        all_accuracy[i - first_person, test_index] = accuracy
        all_discarded[i - first_person, test_index] = 100 * discarded / total
        cm_sum += confusion_matrix(y_test[accepted_mask], predictions[accepted_mask],
                                   labels=np.arange(len(CLASS_NAMES)))

        # --- I due stadi presi separatamente, senza soglia ----------------------
        # Lo stadio 2 viene valutato sulle sole finestre davvero attive, cioe' come se lo
        # stadio 1 non sbagliasse mai: e' il suo risultato al netto della serie.
        active_test = y_test > 0
        acc1 = balanced_accuracy_score(active_test.astype(int), (p_active >= 0.5).astype(int))
        acc2 = balanced_accuracy_score(y_test[active_test] - 1,
                                       np.argmax(p_class[active_test], axis=1))
        all_stage1[i - first_person, test_index] = acc1
        all_stage2[i - first_person, test_index] = acc2

        print(f"  Run test: {test_runs[0]} + {test_runs[1]} | Threshold: {threshold:.2f} | "
              f"Decisione: {DECISION} | Riallineamento: {RECENTER}")
        print(f"  Training: {n_rest} finestre di riposo / {n_active} di movimento")
        print(f"  Campioni: {total} tot / {accepted} accettati / {discarded} scartati "
              f"({100*discarded/total:.1f}%)")
        print(f"  Stadio 1 riposo/attivo: {acc1*100:5.2f}% | "
              f"score interno {grid1.best_score_*100:.2f}% | {grid1.best_params_}")
        print(f"  Stadio 2 quattro mosse: {acc2*100:5.2f}% | "
              f"score interno {grid2.best_score_*100:.2f}% | {grid2.best_params_}")
        print(f"  Catena completa sugli accettati: {accuracy*100:.2f}%")
        print(f"  Tempo: {time.time() - test_start:.1f}s\n")

    patient_mean = np.nanmean(all_accuracy[i - first_person])
    patient_discarded = np.nanmean(all_discarded[i - first_person])
    print(f"  Paziente {subject}: {patient_mean*100:.2f}% con {patient_discarded:.1f}% di scarti "
          f"| Tempo: {time.time() - subject_start:.1f}s\n")

# --- Riepilogo ---------------------------------------------------------------

n_folds = int(np.count_nonzero(~np.isnan(all_accuracy)))
n_falliti = all_accuracy.size - n_folds
print("=" * 60)
print(f"Fold riusciti:          {n_folds}/{all_accuracy.size}"
      + (f" ({n_falliti} falliti, esclusi dalle medie)" if n_falliti else ""))
print(f"Stadio 1 da solo:       {np.nanmean(all_stage1)*100:.2f}%   (riposo vs attivo)")
print(f"Stadio 2 da solo:       {np.nanmean(all_stage2)*100:.2f}%   (quattro movimenti, "
      f"stadio 1 supposto perfetto)")
print(f"Prodotto dei due:       {np.nanmean(all_stage1)*np.nanmean(all_stage2)*100:.2f}%   "
      f"(stima grossolana di quanto costa la serie)")
print(f"Catena completa:        {np.nanmean(all_accuracy)*100:.2f}%   con "
      f"{np.nanmean(all_discarded):.1f}% di campioni scartati")
print(f"Deviazione standard:    {np.nanstd(all_accuracy)*100:.2f}%")
print(f"Caso su cinque classi:  {100/len(CLASS_NAMES):.2f}%")
print()
print("Matrice di confusione media (righe = vero, colonne = predetto):")
cm = cm_sum / n_folds
print(f"{'':>11s}" + "".join(f"{name:>11s}" for name in CLASS_NAMES))
for name, row in zip(CLASS_NAMES, cm):
    print(f"{name:>11s}" + "".join(f"{v:>11.1f}" for v in row))


Feature per finestra: 135 (3 bande x 9 canali)

Paziente 001
  Run test: 12 + 14 | Threshold: 0.70 | Decisione: cascata | Riallineamento: per_run
  Training: 508 finestre di riposo / 336 di movimento
  Campioni: 422 tot / 304 accettati / 118 scartati (28.0%)
  Stadio 1 riposo/attivo: 65.51% | score interno 69.37% | {'clf__shrinkage': 0.5}
  Stadio 2 quattro mosse: 68.45% | score interno 44.58% | {'clf__shrinkage': 0.5}
  Catena completa sugli accettati: 53.26%
  Tempo: 14.0s

  Run test: 8 + 10 | Threshold: 0.65 | Decisione: cascata | Riallineamento: per_run
  Training: 508 finestre di riposo / 336 di movimento
  Campioni: 422 tot / 296 accettati / 126 scartati (29.9%)
  Stadio 1 riposo/attivo: 65.81% | score interno 70.83% | {'clf__shrinkage': 0.5}
  Stadio 2 quattro mosse: 57.74% | score interno 55.28% | {'clf__shrinkage': 'auto'}
  Catena completa sugli accettati: 49.13%
  Tempo: 0.4s

  Run test: 4 + 6 | Threshold: 0.65 | Decisione: cascata | Riallineamento: per_run
  Training: 508